# SAIR IC50 Oracle — Colab Training

Trains the IC50 oracle (ESM2 mean-pool protein encoder + ChemBERTa chemical encoder + protein-family embedding) on Colab GPU.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Pro)
2. Make sure mean-pooled ESM embeddings exist in `data/esm_embeddings/`  
   (generated by `scripts/precompute_esm.py` — without `--residue-dir`)
3. Run all cells top-to-bottom — Cell 5 will generate `data/chem_emb_cache.pt` automatically on first run
4. If the session disconnects: re-run cells 1–5, then set `RESUME` in Cell 7

## Cell 1 — Verify GPU and mount Google Drive

In [1]:
import torch

assert torch.cuda.is_available(), "No GPU found — change runtime type to GPU before continuing."
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

PyTorch : 2.10.0+cu128
CUDA    : 12.8
GPU     : NVIDIA A100-SXM4-40GB
VRAM    : 42.4 GB
Mounted at /content/drive


## Cell 2 — Install packages

PyTorch Geometric compiled extensions must match the exact PyTorch + CUDA version —
the URL is computed dynamically. If they fail, PyG falls back to pure-PyTorch scatter
ops: GINEConv still works correctly for backward compat, just ~20% slower.

`transformers` provides ChemBERTa — the backbone is frozen and used only during
precomputation (`scripts/precompute_ligands.py --chemberta-output`), but the package
must be present at import time.

In [2]:
import subprocess, sys, torch

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = "cu" + torch.version.cuda.replace('.', '')
pyg_url   = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"

print(f"PyTorch {torch_ver}  |  {cuda_tag}")
print(f"PyG wheel URL: {pyg_url}\n")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

print("Installing torch_geometric...")
pip("torch_geometric")

print("Installing compiled PyG extensions (torch_scatter, torch_sparse)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch_scatter", "torch_sparse", "-f", pyg_url]
)
if result.returncode != 0:
    print("⚠  Compiled extensions unavailable — using pure-PyTorch fallback (correct, ~20% slower)")
else:
    print("✓  Compiled extensions installed")

# transformers 5.x requires torch>=2.6; pin to 4.x for Colab's current torch 2.5.x
print("Installing rdkit, psutil, wandb, transformers...")
pip("rdkit", "psutil", "wandb", "transformers>=4.40,<5.0")

print("\n✓ All packages ready")

PyTorch 2.10.0  |  cu128
PyG wheel URL: https://data.pyg.org/whl/torch-2.10.0+cu128.html

Installing torch_geometric...
Installing compiled PyG extensions (torch_scatter, torch_sparse)...
✓  Compiled extensions installed
Installing rdkit, psutil, wandb, transformers...

✓ All packages ready


## Cell 3 — Set working directory and Python path

In [3]:
import sys, os

PROJECT = '/content/drive/MyDrive/sair-ic50-oracle'
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

required = {
    'data/sair.parquet'                 : os.path.exists,
    'data/esm_embeddings/'              : os.path.isdir,
    'data/splits/train.txt'             : os.path.exists,
    'data/splits/val.txt'               : os.path.exists,
    'data/long_protein_annotations.csv' : os.path.exists,
    'config/default.yaml'               : os.path.exists,
}
optional = {
    'data/chem_emb_cache.pt': os.path.exists,
}

all_ok = True
for path, check in required.items():
    ok = check(path)
    print(f"{'✓' if ok else '✗'}  {path}")
    all_ok = all_ok and ok

for path, check in optional.items():
    ok = check(path)
    print(f"{'✓' if ok else '⚠'} (optional) {path}")
    if not ok:
        print(f"   → chem_emb_cache.pt missing: ChemBERTa embeddings will be computed "
              f"in memory at dataset init (~5 min, GPU recommended).")
        print(f"   → Generate offline: python scripts/precompute_ligands.py "
              f"--parquet data/sair.parquet --chemberta-output data/chem_emb_cache.pt")

if not all_ok:
    raise RuntimeError("One or more required files are missing. See README Quickstart.")

from oracle.dataset import SAIRDataset
from oracle.model   import IC50Oracle
print("\n✓ Project imports OK")

✓  data/sair.parquet
✓  data/esm_embeddings/
✓  data/splits/train.txt
✓  data/splits/val.txt
✓  data/long_protein_annotations.csv
✓  config/default.yaml
✓ (optional) data/chem_emb_cache.pt

✓ Project imports OK


## Cell 4 — Stage data to local SSD

Google Drive FUSE adds ~100 ms overhead **per file**. This cell copies everything
to Colab's local SSD (`/tmp/sair/`) once per session.

- `sair.parquet` — large sequential read benefits from SSD
- `esm_embeddings/` — 3,563 × .pt files (mean-pooled [1280] float32 tensors, ~17 MB)
- `chem_emb_cache.pt` — precomputed ChemBERTa CLS tokens for all unique SMILES  
  Generate once: `python scripts/precompute_ligands.py --parquet data/sair.parquet --chemberta-output data/chem_emb_cache.pt`

Checkpoints, annotations, and splits paths are set to absolute Drive paths so they
persist across sessions even if this cell is re-run before Cell 3 sets the CWD.

**Re-run this cell at the start of every new session** (local SSD is wiped on disconnect).

In [4]:
import shutil, os, time, yaml

SSD = '/tmp/sair'
os.makedirs(SSD, exist_ok=True)

items = [
    ('data/sair.parquet',   f'{SSD}/sair.parquet',   False, 'sair.parquet'),
    ('data/esm_embeddings', f'{SSD}/esm_embeddings',  True,  'ESM mean-pool embeddings'),
]
# Stage ChemBERTa cache only if it exists
if os.path.exists('data/chem_emb_cache.pt'):
    items.append(('data/chem_emb_cache.pt', f'{SSD}/chem_emb_cache.pt', False, 'ChemBERTa embedding cache'))

t0 = time.time()
for src, dst, is_dir, label in items:
    if os.path.exists(dst):
        print(f"  ✓ {label} — already on SSD, skipping")
        continue
    print(f"  Copying {label} ...", end='', flush=True)
    t = time.time()
    shutil.copytree(src, dst) if is_dir else shutil.copy2(src, dst)
    print(f" {time.time()-t:.0f}s")

with open('config/default.yaml') as f:
    cfg = yaml.safe_load(f)

# Fast paths: read large files from SSD
cfg['paths']['parquet']        = f'{SSD}/sair.parquet'
cfg['paths']['esm_cache']      = f'{SSD}/esm_embeddings/'
if os.path.exists(f'{SSD}/chem_emb_cache.pt'):
    cfg['paths']['chem_emb_cache'] = f'{SSD}/chem_emb_cache.pt'
else:
    cfg['paths']['chem_emb_cache'] = None  # fall back to in-memory computation (~5 min)

# Absolute Drive paths for outputs and small read-only files — these must resolve
# to Google Drive regardless of whether CWD was set correctly after a reconnect.
cfg['paths']['checkpoints'] = f'{PROJECT}/{cfg["paths"]["checkpoints"]}'
cfg['paths']['annotations'] = f'{PROJECT}/data/long_protein_annotations.csv'
cfg['paths']['splits']      = f'{PROJECT}/data/splits/'

SESSION_CONFIG = f'{SSD}/colab_config.yaml'
with open(SESSION_CONFIG, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"\n✓ Data staged in {time.time()-t0:.0f}s")
print(f"  Parquet        : {SSD}/sair.parquet")
print(f"  ESM mean-pool  : {SSD}/esm_embeddings/")
if os.path.exists(f'{SSD}/chem_emb_cache.pt'):
    print(f"  ChemBERTa      : {SSD}/chem_emb_cache.pt")
else:
    print(f"  ChemBERTa      : not found — will compute in memory at dataset init")
print(f"  Config         : {SESSION_CONFIG}")
print(f"  Checkpoints    : {cfg['paths']['checkpoints']} (Google Drive — persistent)")

  Copying sair.parquet ... 9s
  Copying ESM mean-pool embeddings ... 75s
  Copying ChemBERTa embedding cache ... 43s

✓ Data staged in 131s
  Parquet        : /tmp/sair/sair.parquet
  ESM mean-pool  : /tmp/sair/esm_embeddings/
  ChemBERTa      : /tmp/sair/chem_emb_cache.pt
  Config         : /tmp/sair/colab_config.yaml
  Checkpoints    : /content/drive/MyDrive/sair-ic50-oracle/checkpoints/chemberta/ (Google Drive — persistent)


## Cell 5 — Precompute ChemBERTa embeddings (once per dataset)

Runs `precompute_ligands.py` on the Colab GPU and writes the cache directly to Google Drive — **survives session disconnects**.  Skip automatically if the file already exists.

- Input : `sair.parquet` from local SSD  
- Output: `data/chem_emb_cache.pt` on Drive (~2.3 GB)  
- Time  : ~5–10 min on A100 (734 k SMILES, batch=256)

In [5]:
import os, shutil, time, yaml
from pathlib import Path
import importlib.util, pandas as pd

CACHE_DRIVE = Path(f'{PROJECT}/data/chem_emb_cache.pt')

if CACHE_DRIVE.exists():
    print(f'✓ ChemBERTa cache already exists on Drive — skipping precomputation.')
else:
    # Import build_chemberta_cache directly so tqdm.auto renders as a Colab widget
    # (subprocess falls back to line-by-line output with no live bar)
    spec = importlib.util.spec_from_file_location(
        'precompute_ligands', f'{PROJECT}/scripts/precompute_ligands.py'
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    parquet_src = f'{SSD}/sair.parquet' if os.path.exists(f'{SSD}/sair.parquet') \
                  else f'{PROJECT}/data/sair.parquet'

    df = pd.read_parquet(parquet_src, columns=['SMILES'])
    unique_smiles = df['SMILES'].dropna().unique().tolist()
    del df
    print(f'Unique SMILES: {len(unique_smiles):,}  |  output → {CACHE_DRIVE}\n')

    mod.build_chemberta_cache(unique_smiles, CACHE_DRIVE)
    print(f'\n✓ ChemBERTa cache saved to Drive.')

# Stage to SSD so training reads it at full speed
SSD_CACHE = f'{SSD}/chem_emb_cache.pt'
if not os.path.exists(SSD_CACHE):
    print('Staging cache to SSD ...', end='', flush=True)
    t = time.time()
    shutil.copy2(str(CACHE_DRIVE), SSD_CACHE)
    print(f' {time.time()-t:.0f}s')
else:
    print('✓ Cache already on SSD')

# Patch session config so train.py reads from SSD
with open(SESSION_CONFIG) as f:
    cfg = yaml.safe_load(f)
cfg['paths']['chem_emb_cache'] = SSD_CACHE
with open(SESSION_CONFIG, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f'✓ Session config: chem_emb_cache → {SSD_CACHE}')


✓ ChemBERTa cache already exists on Drive — skipping precomputation.
✓ Cache already on SSD
✓ Session config: chem_emb_cache → /tmp/sair/chem_emb_cache.pt


## Cell 6 — (Optional) wandb login

Skip this cell to run without wandb logging.  
To enable: paste your API key when prompted, or pre-set the `WANDB_API_KEY` env var.

In [6]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: valentin_badea (valentin_badea-harvard-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Cell 7 — Train

Calls `train()` directly so `tqdm.auto` renders a live progress bar widget.

Dataset init takes ~10–20 s (mean-pool embeddings + ChemBERTa cache load from SSD).  
Each epoch shows a tqdm bar with live loss and lr.  The model predicts in normalised
pIC50 space; metrics printed each epoch are already denormalised to raw pIC50 units.

**After a disconnection:** re-run cells 1–5, then set `RESUME` to the checkpoint path
printed by Cell 4 (e.g. `checkpoints/chemberta/best.pt`).

In [7]:
import yaml
from scripts.train import train, load_config

SESSION_CONFIG = '/tmp/sair/colab_config.yaml'
RESUME         = None   # set to checkpoint path to resume after disconnect
                        # e.g. RESUME = '/content/drive/MyDrive/sair-ic50-oracle/checkpoints/chemberta/best.pt'

config = load_config(SESSION_CONFIG)
train(config, resume=RESUME)

Device: cuda
Mixed precision: torch.autocast({'device_type': 'cuda', 'dtype': torch.float16})
Split sizes — train: 298,888  val: 41,269
Protein filter loaded: 3,563 proteins allowed
Protein filter: 2,254,568 -> 2,032,088 rows (dropped 222,480)
Dataset ready: 298,888 samples across 2,859 proteins
Pre-loading 2,859 mean-pooled ESM embeddings ... done (7 MB)
Loading ChemBERTa cache from /tmp/sair/chem_emb_cache.pt ... done (221,828 entries)
Protein filter loaded: 3,563 proteins allowed
Protein filter: 2,254,568 -> 2,032,088 rows (dropped 222,480)
Dataset ready: 41,269 samples across 354 proteins
Pre-loading 354 mean-pooled ESM embeddings ... done (1 MB)
Loading ChemBERTa cache from /tmp/sair/chem_emb_cache.pt ... done (37,992 entries)
Model parameters: 1,320,801


Epoch 1:   0%|          | 0/2336 [00:00<?, ?it/s]

Epoch   1 | train_loss=0.6891 | val_spearman=0.3561 | val_mae=1.0221 | per_prot_spearman=0.2021
  Checkpoint saved (per_prot_spearman=0.3561)


Epoch 2:   0%|          | 0/2336 [00:00<?, ?it/s]

Epoch   2 | train_loss=0.5899 | val_spearman=0.3610 | val_mae=1.0275 | per_prot_spearman=0.1956
  Checkpoint saved (per_prot_spearman=0.3610)


...

Epoch  20 | train_loss=0.3922 | val_spearman=0.3663 | val_mae=1.0509 | per_prot_spearman=0.2162


Epoch 21:   0%|          | 0/2336 [00:00<?, ?it/s]

Epoch  21 | train_loss=0.3877 | val_spearman=0.3742 | val_mae=1.0293 | per_prot_spearman=0.1937
Early stopping after 15 epochs without improvement.


train/epoch_loss,█▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train/loss,█▆█▅▅▅█▄▆▆▆▅▅▃▅▂▄▄▅▄▃▃▃▂▃▄▃▅▃▃▃▃▃▂▃▁▂▂▂▃
train/lr,███████████▇▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▂▂▁
val/mae,▄▅▄█▅▁▄▂█▄▄▅▅▄▄▃▅█▅█▅
val/pearson,▃▄▆▁▅█▅▆▃▇▅▆▇▆▆▆▅▅▅▅▅
val/per_protein_spearman,▄▃▃▁▃▅▆▂▂▅▆█▅▅▅▆▅▅▆▆▃
val/spearman,▃▄▆▁▅█▅▆▃▆▅▆▇▆▆▅▅▄▅▄▅
train/epoch_loss,0.38771
train/loss,0.40415
train/lr,6e-05
val/mae,1.0293



Done. Best per_prot_spearman: 0.4088
Checkpoints in: /content/drive/MyDrive/sair-ic50-oracle/checkpoints/chemberta


## Cell 8 — Checkpoint inspection

Run after training completes (or any time) to see the best checkpoint metrics.

In [9]:
import torch, yaml

with open('/tmp/sair/colab_config.yaml') as f:
    cfg = yaml.safe_load(f)
ckpt_dir = cfg['paths']['checkpoints']

ckpt = torch.load(f'{ckpt_dir}/best.pt', map_location='cpu', weights_only=False)
print(f"Checkpoint dir          : {ckpt_dir}")
print(f"Best epoch              : {ckpt['epoch']}")
print(f"Per-prot Spearman (best): {ckpt['best_spearman']:.4f}")
print(f"Val Spearman            : {ckpt['val_spearman']:.4f}")
print(f"Val MAE                 : {ckpt['val_mae']:.4f}")
print(f"Global step             : {ckpt['global_step']:,}")
print(f"Epochs w/o improvement  : {ckpt.get('epochs_without_improvement', '?')}")

Checkpoint dir          : /content/drive/MyDrive/sair-ic50-oracle/checkpoints/chemberta/
Best epoch              : 6
Per-prot Spearman (best): 0.4088
Val Spearman            : 0.4088
Val MAE                 : 0.9932
Global step             : 14,016
Epochs w/o improvement  : 0


---
## Reference: expected timings

| Phase | T4 (free) | A100 (Pro) |
|---|---|---|
| Data staging — Cell 4 (once per session) | ~60–90 s | ~60–90 s |
| ChemBERTa precompute — Cell 5 (**once ever**) | ~20 min | ~5–10 min |
| Dataset init (mean-pool + ChemBERTa cache load) | ~20 s | ~20 s |
| Training per epoch | ~3–4 min | ~1–2 min |
| Full run with patience=15 (~30 epochs) | ~2 hrs | ~45 min |

## Reference: reconnection checklist

After a session disconnects, `/tmp/` is wiped and packages are uninstalled. On reconnect:

1. Cell 1 — GPU check + Drive mount
2. Cell 2 — reinstall packages
3. Cell 3 — set paths + imports
4. Cell 4 — re-stage data to SSD
5. Cell 5 — ChemBERTa precompute (skips automatically if cache exists on Drive)
6. Cell 7 — set `RESUME = '<checkpoint path>/best.pt'` and run

Checkpoints and the ChemBERTa cache both survive in Google Drive.